<a href="https://colab.research.google.com/github/tratramcute/Datathon-R1/blob/main/2.%20Tomorrow%20Business%20Analyst%20(Predictive%20%26%20Prescriptive).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#2. EDA - PREDICTIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Analyze regional revenue, profit trend and profitability

In [ ]:
import pandas as pd
import numpy as np

dataset_path = '/content/drive/MyDrive/dataset/'

order_items = pd.read_csv(dataset_path + 'order_items.csv')
customers = pd.read_csv(dataset_path + 'customers.csv')
products = pd.read_csv(dataset_path + 'products.csv')
payments = pd.read_csv(dataset_path + 'payments.csv')
reviews = pd.read_csv(dataset_path + 'reviews.csv')
returns = pd.read_csv(dataset_path + 'returns.csv')
sales = pd.read_csv(dataset_path + 'sales.csv', parse_dates=['Date'])

# Use reviews as bridge: customer_id + order_id
# Then customers -> geography via zip
geography = pd.read_csv(dataset_path + 'geography.csv')

# reviews: order_id, customer_id, review_date
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
reviews['year'] = reviews['review_date'].dt.year

# customer -> region
cust_region = customers[['customer_id','zip']].merge(geography[['zip','region']].drop_duplicates(), on='zip', how='left')

# order -> customer -> region
order_region = reviews[['order_id','customer_id','review_date','year']].drop_duplicates('order_id')\
    .merge(cust_region, on='customer_id', how='left')

# order -> payment value
order_pay = order_region.merge(payments[['order_id','payment_value']], on='order_id', how='left')

# order -> profitability: revenue - cogs via order_items + products
oi = order_items.merge(products[['product_id','cogs']], on='product_id', how='left')
oi['line_revenue'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']
oi['line_cogs'] = oi['cogs'] * oi['quantity']
oi['line_profit'] = oi['line_revenue'] - oi['line_cogs']
order_profit = oi.groupby('order_id')[['line_revenue','line_cogs','line_profit']].sum().reset_index()
order_profit['is_profitable'] = (order_profit['line_profit'] > 0).astype(int)

# Full order table
full = order_pay.merge(order_profit, on='order_id', how='left')
full = full[full['year'].between(2012, 2022)]

print("=== ANNUAL ORDERS BY REGION ===")
annual_region = full.groupby(['year','region']).agg(
    orders=('order_id','count'),
    revenue=('line_revenue','sum'),
    profit=('line_profit','sum'),
    profitable_orders=('is_profitable','sum')
).reset_index()
annual_region['profit_margin'] = annual_region['profit'] / annual_region['revenue']
annual_region['profitable_pct'] = annual_region['profitable_orders'] / annual_region['orders']
print(annual_region[annual_region['year'] >= 2018].to_string())

/tmp/ipykernel_9123/561823818.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(dataset_path + 'order_items.csv')


=== ANNUAL ORDERS BY REGION ===
    year   region  orders       revenue        profit  profitable_orders  profit_margin  profitable_pct
18  2018  Central    3636  9.188851e+07  1.162784e+07               2822       0.126543        0.776128
19  2018     East    5273  1.323082e+08  1.679484e+07               3999       0.126937        0.758392
20  2018     West    3119  7.366466e+07  8.736293e+06               2401       0.118595        0.769798
21  2019  Central    2225  5.950889e+07  4.166755e+06               1498       0.070019        0.673258
22  2019     East    3158  8.131739e+07  5.579249e+06               2101       0.068611        0.665294
23  2019     West    1715  4.190934e+07  3.193175e+06               1187       0.076192        0.692128
24  2020  Central    1898  5.604681e+07  6.494969e+06               1421       0.115885        0.748683
25  2020     East    2692  7.932745e+07  8.908989e+06               1992       0.112307        0.739970
26  2020     West    1422  3.897

## 2. Trend Extrapolation, Seasonality, and Return Rate Analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

dataset_path = '/content/drive/MyDrive/dataset/'

order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
customers = pd.read_csv(dataset_path + 'customers.csv')
products = pd.read_csv(dataset_path + 'products.csv')
payments = pd.read_csv(dataset_path + 'payments.csv')
reviews = pd.read_csv(dataset_path + 'reviews.csv')
returns = pd.read_csv(dataset_path + 'returns.csv')
geography = pd.read_csv(dataset_path + 'geography.csv')
sales = pd.read_csv(dataset_path + 'sales.csv', parse_dates=['Date'])

reviews['review_date'] = pd.to_datetime(reviews['review_date'])
reviews['year'] = reviews['review_date'].dt.year
reviews['month'] = reviews['review_date'].dt.month

cust_region = customers[['customer_id','zip']].merge(geography[['zip','region']].drop_duplicates(), on='zip', how='left')
order_region = reviews[['order_id','customer_id','review_date','year','month']].drop_duplicates('order_id')\
    .merge(cust_region, on='customer_id', how='left')

oi = order_items.merge(products[['product_id','cogs','segment','category']], on='product_id', how='left')
oi['line_revenue'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']
oi['line_cogs'] = oi['cogs'] * oi['quantity']
oi['line_profit'] = oi['line_revenue'] - oi['line_cogs']
order_profit = oi.groupby('order_id')[['line_revenue','line_cogs','line_profit']].sum().reset_index()

full = order_region.merge(order_profit, on='order_id', how='left')
full = full[full['year'].between(2012, 2022)]

# === TREND EXTRAPOLATION per region ===
print("=== TREND EXTRAPOLATION: Orders & Profit 2023-2025 ===")
for region in ['East','West','Central']:
    df_r = full[full['region']==region].groupby('year').agg(
        orders=('order_id','count'),
        profit=('line_profit','sum')
    ).reset_index()

    # Linear regression on orders
    slope_o, intercept_o, r_o, _, _ = stats.linregress(df_r['year'], df_r['orders'])
    slope_p, intercept_p, r_p, _, _ = stats.linregress(df_r['year'], df_r['profit'])

    print(f"\n{region}:")
    print(f"  Orders trend: {slope_o:+.0f} đơn/năm (R²={r_o**2:.2f})")
    print(f"  Profit trend: {slope_p:+.0f}/năm (R²={r_p**2:.2f})")
    for yr in [2023, 2024, 2025]:
        pred_o = intercept_o + slope_o * yr
        pred_p = intercept_p + slope_p * yr
        print(f"  {yr}: ~{pred_o:.0f} đơn, profit ~{pred_p/1e6:.2f}M")

# === SEASONALITY ===
print("\n\n=== SEASONALITY: Peak months by region ===")
monthly = full.groupby(['region','month']).agg(
    avg_orders=('order_id','count'),
    avg_profit=('line_profit','sum')
).reset_index()
for region in ['East','West','Central']:
    df_m = monthly[monthly['region']==region].sort_values('avg_orders', ascending=False)
    top3 = df_m.head(3)['month'].tolist()
    bot3 = df_m.tail(3)['month'].tolist()
    print(f"{region} — Peak months: {top3} | Low months: {bot3}")

# === RETURN RATE by region + segment ===
print("\n\n=== RETURN RATE by region ===")
order_ret = returns.merge(order_region[['order_id','region']], on='order_id', how='left')
ret_region = order_ret.groupby('region')['return_id'].count().reset_index(name='returns')
ord_region = full.groupby('region')['order_id'].count().reset_index(name='orders')
rr = ret_region.merge(ord_region, on='region')
rr['return_rate'] = rr['returns'] / rr['orders']
print(rr)

# Return rate by region + segment
order_seg = oi.groupby('order_id')['segment'].first().reset_index()
order_ret2 = returns.merge(order_region[['order_id','region']], on='order_id', how='left')\
    .merge(order_seg, on='order_id', how='left')
rr2 = order_ret2.groupby(['region','segment'])['return_id'].count().reset_index(name='returns')
ord_seg = full.merge(order_seg, on='order_id', how='left').groupby(['region','segment'])['order_id'].count().reset_index(name='orders')
rr_seg = rr2.merge(ord_seg, on=['region','segment'])
rr_seg['return_rate'] = rr_seg['returns'] / rr_seg['orders']
print("\nReturn rate by region + segment:")
print(rr_seg.sort_values(['region','return_rate'], ascending=[True,False]).to_string())

=== TREND EXTRAPOLATION: Orders & Profit 2023-2025 ===

East:
  Orders trend: -286 đơn/năm (R²=0.26)
  Profit trend: -749028/năm (R²=0.29)
  2023: ~2848 đơn, profit ~6.23M
  2024: ~2562 đơn, profit ~5.49M
  2025: ~2276 đơn, profit ~4.74M

West:
  Orders trend: -214 đơn/năm (R²=0.30)
  Profit trend: -490687/năm (R²=0.33)
  2023: ~1431 đơn, profit ~3.19M
  2024: ~1217 đơn, profit ~2.70M
  2025: ~1004 đơn, profit ~2.21M

Central:
  Orders trend: -123 đơn/năm (R²=0.16)
  Profit trend: -343877/năm (R²=0.14)
  2023: ~2111 đơn, profit ~4.70M
  2024: ~1988 đơn, profit ~4.36M
  2025: ~1865 đơn, profit ~4.01M


=== SEASONALITY: Peak months by region ===
East — Peak months: [5, 6, 7] | Low months: [3, 11, 2]
West — Peak months: [6, 4, 5] | Low months: [3, 11, 2]
Central — Peak months: [5, 6, 7] | Low months: [11, 12, 2]


=== RETURN RATE by region ===
Empty DataFrame
Columns: [region, returns, orders, return_rate]
Index: []

Return rate by region + segment:
Empty DataFrame
Columns: [region, segme

## 3. Fix dtype and recompute return rate

## 4. Quantify return rate trend, stockout revenue loss, category order trend

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

dataset_path = '/content/drive/MyDrive/dataset/'

order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
products = pd.read_csv(dataset_path + 'products.csv')
returns = pd.read_csv(dataset_path + 'returns.csv')
inventory = pd.read_csv(dataset_path + 'inventory.csv')

# ============================================================
# 1. RETURN RATE TREND BY SEGMENT OVER TIME
# ============================================================
ret_prod = returns.merge(products[['product_id','segment','category']], on='product_id', how='left')
ret_prod['return_date'] = pd.to_datetime(ret_prod['return_date'])
ret_prod['year'] = ret_prod['return_date'].dt.year

# Annual returns by segment
ret_annual = ret_prod.groupby(['year','segment'])['return_id'].count().reset_index(name='returns')

# Annual orders by segment
oi_prod = order_items.merge(products[['product_id','segment']], on='product_id', how='left')
# Need order date — use return_date as proxy won't work, use inventory year as reference
# Use reviews for order date
reviews = pd.read_csv(dataset_path + 'reviews.csv')
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
reviews['year'] = reviews['review_date'].dt.year
oi_year = oi_prod.merge(reviews[['order_id','year']].drop_duplicates('order_id'), on='order_id', how='left')
orders_annual = oi_year.groupby(['year','segment'])['order_id'].nunique().reset_index(name='orders')

rr_annual = ret_annual.merge(orders_annual, on=['year','segment'], how='left')
rr_annual['return_rate'] = rr_annual['returns'] / rr_annual['orders']

print("=== RETURN RATE TREND: Balanced & Everyday (2018-2022) ===")
for seg in ['Balanced','Everyday','Activewear']:
    df = rr_annual[rr_annual['segment']==seg].sort_values('year')
    print(f"\n{seg}:")
    print(df[df['year']>=2018][['year','returns','orders','return_rate']].to_string(index=False))
    # Trend
    df_clean = df[df['year']>=2015].dropna()
    if len(df_clean) > 2:
        slope, intercept, r, _, _ = stats.linregress(df_clean['year'], df_clean['return_rate'])
        for yr in [2023, 2025]:
            pred = intercept + slope * yr
            print(f"  → Predicted {yr}: {pred:.1%}")

# ============================================================
# 2. INVENTORY: QUANTIFY STOCKOUT + OVERSTOCK IMPACT
# ============================================================
print("\n\n=== INVENTORY ANALYSIS ===")
inv = inventory.copy()

# Annual stockout/overstock flags
inv_annual = inv.groupby('year').agg(
    total_records=('product_id','count'),
    stockout_records=('stockout_flag','sum'),
    overstock_records=('overstock_flag','sum'),
    avg_fill_rate=('fill_rate','mean'),
    avg_stockout_days=('stockout_days','mean'),
    total_units_sold=('units_sold','sum'),
    total_stock_on_hand=('stock_on_hand','sum')
).reset_index()
inv_annual['stockout_pct'] = inv_annual['stockout_records'] / inv_annual['total_records']
inv_annual['overstock_pct'] = inv_annual['overstock_records'] / inv_annual['total_records']
print(inv_annual[['year','stockout_pct','overstock_pct','avg_fill_rate','avg_stockout_days']].to_string(index=False))

# ============================================================
# 3. REVENUE LOST TO STOCKOUT (leading indicator)
# ============================================================
print("\n=== ESTIMATED REVENUE LOST TO STOCKOUT ===")
# avg daily sales * stockout days per product per month
inv['avg_daily_sales'] = inv['units_sold'] / 30
inv['lost_units'] = inv['avg_daily_sales'] * inv['stockout_days']

# Join with product price
inv_price = inv.merge(products[['product_id','price']], on='product_id', how='left')
inv_price['lost_revenue'] = inv_price['lost_units'] * inv_price['price']

lost_annual = inv_price.groupby('year').agg(
    lost_revenue=('lost_revenue','sum'),
    total_units_sold=('units_sold','sum')
).reset_index()
lost_annual['lost_rev_M'] = lost_annual['lost_revenue'] / 1e6

print(lost_annual[['year','lost_rev_M','total_units_sold']].to_string(index=False))

# ============================================================
# 4. TREND: Orders by category
# ============================================================
print("\n=== ORDER TREND BY CATEGORY ===")
oi_cat = order_items.merge(products[['product_id','category']], on='product_id', how='left')
oi_cat_year = oi_cat.merge(reviews[['order_id','year']].drop_duplicates('order_id'), on='order_id', how='left')
cat_annual = oi_cat_year.groupby(['year','category'])['order_id'].nunique().reset_index(name='orders')

for cat in ['Streetwear','Outdoor','Casual','GenZ']:
    df = cat_annual[cat_annual['category']==cat].sort_values('year')
    df_clean = df[df['year']>=2015].dropna()
    if len(df_clean) > 2:
        slope, intercept, r, _, _ = stats.linregress(df_clean['year'], df_clean['orders'])
        print(f"\n{cat}: trend {slope:+.0f} đơn/năm (R²={r**2:.2f})")
        for yr in [2023, 2025]:
            print(f"  → {yr}: ~{intercept + slope*yr:.0f} đơn")
        print(f"  2022 actual: {df[df['year']==2022]['orders'].values}")

=== RETURN RATE TREND: Balanced & Everyday (2018-2022) ===

Balanced:
 year  returns  orders  return_rate
 2018      588    1784     0.329596
 2019      470    1377     0.341322
 2020      441    1385     0.318412
 2021      501    1391     0.360173
 2022      574    1609     0.356743
  → Predicted 2023: 35.0%
  → Predicted 2025: 35.6%

Everyday:
 year  returns  orders  return_rate
 2018     1151    3439     0.334690
 2019      625    1747     0.357756
 2020      525    1416     0.370763
 2021      489    1294     0.377898
 2022      411    1143     0.359580
  → Predicted 2023: 37.7%
  → Predicted 2025: 38.7%

Activewear:
 year  returns  orders  return_rate
 2018     1070    2910     0.367698
 2019      642    1787     0.359261
 2020      487    1345     0.362082
 2021      419    1144     0.366259
 2022      416    1018     0.408644
  → Predicted 2023: 38.4%
  → Predicted 2025: 39.0%


=== INVENTORY ANALYSIS ===
 year  stockout_pct  overstock_pct  avg_fill_rate  avg_stockout_days
 201

## 5. Quantify revenue impact of reducing return rate and improving fill rate

In [ ]:
import pandas as pd
import numpy as np

dataset_path = '/content/drive/MyDrive/dataset/'

products = pd.read_csv(dataset_path + 'products.csv')
order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
returns = pd.read_csv(dataset_path + 'returns.csv')
inventory = pd.read_csv(dataset_path + 'inventory.csv')

# ============================================================
# GOAL: Nếu giảm return rate xuống mức target, tăng được bao nhiêu revenue?
# ============================================================
ret_prod = returns.merge(products[['product_id','segment','category','price']], on='product_id', how='left')

# Current return rate by segment (overall)
ret_by_seg = ret_prod.groupby('segment').agg(
    returns=('return_id','count'),
    avg_refund=('refund_amount','mean'),
    total_refund=('refund_amount','sum')
).reset_index()

oi_seg = order_items.merge(products[['product_id','segment']], on='product_id')\
    .groupby('segment')['order_id'].nunique().reset_index(name='orders')

rr = ret_by_seg.merge(oi_seg, on='segment')
rr['current_rr'] = rr['returns'] / rr['orders']

# Target: reduce return rate to industry benchmark ~15% for fashion ecommerce
# If return rate drops from current to 15%, how many fewer returns?
# Revenue saved = fewer_returns * avg_refund
TARGET_RR = 0.15
rr['target_rr'] = TARGET_RR
rr['returns_saved'] = ((rr['current_rr'] - TARGET_RR) * rr['orders']).clip(lower=0)
rr['revenue_saved_M'] = rr['returns_saved'] * rr['avg_refund'] / 1e6

print("=== REVENUE IMPACT IF RETURN RATE → 15% ===")
print(rr[['segment','current_rr','orders','returns','returns_saved','avg_refund','revenue_saved_M']].sort_values('revenue_saved_M', ascending=False).to_string(index=False))
print(f"\nTOTAL revenue saved: {rr['revenue_saved_M'].sum():.2f}M")

# ============================================================
# GOAL: Stockout fix — if fill_rate improves from 96.4% to 99%
# ============================================================
print("\n\n=== STOCKOUT GOAL: Fill Rate 96.4% → 99% ===")
inv_2022 = inventory[inventory['year']==2022]
current_fill = inv_2022['fill_rate'].mean()
target_fill = 0.99
units_sold_2022 = inv_2022['units_sold'].sum()

# Units that could have been sold if fill_rate = 99%
# units_sold / fill_rate = total demand
inv_2022 = inv_2022.merge(products[['product_id','price']], on='product_id', how='left')
inv_2022['implied_demand'] = inv_2022['units_sold'] / inv_2022['fill_rate'].replace(0, np.nan)
inv_2022['additional_units'] = inv_2022['implied_demand'] * (target_fill - inv_2022['fill_rate']).clip(lower=0)
inv_2022['additional_revenue'] = inv_2022['additional_units'] * inv_2022['price']

print(f"Current avg fill rate (2022): {current_fill:.2%}")
print(f"Target fill rate: {target_fill:.2%}")
print(f"Additional revenue if fill_rate → 99%: {inv_2022['additional_revenue'].sum()/1e6:.2f}M")
print(f"Lost revenue to stockout (2022): 15.56M (from prior calc)")

# By category
inv_cat = inv_2022.groupby('category').agg(
    additional_rev=('additional_revenue','sum'),
    avg_fill=('fill_rate','mean'),
    stockout_pct=('stockout_flag','mean')
).reset_index()
inv_cat['additional_rev_M'] = inv_cat['additional_rev'] / 1e6
print("\nBy category:")
print(inv_cat[['category','avg_fill','stockout_pct','additional_rev_M']].sort_values('additional_rev_M', ascending=False).to_string(index=False))

=== REVENUE IMPACT IF RETURN RATE → 15% ===
    segment  current_rr  orders  returns  returns_saved   avg_refund  revenue_saved_M
 Activewear    0.076516  171011    13085            0.0  4981.521045              0.0
All-weather    0.053518   22553     1207            0.0 10603.616156              0.0
   Balanced    0.054483  103023     5613            0.0 27320.109816              0.0
   Everyday    0.055880  181514    10143            0.0 16735.955341              0.0
Performance    0.056509   96303     5442            0.0 14038.838089              0.0
    Premium    0.055649   30944     1722            0.0  8573.078786              0.0
   Standard    0.055016   10924      601            0.0 11993.878253              0.0
     Trendy    0.057376   37054     2126            0.0  5242.543824              0.0

TOTAL revenue saved: 0.00M


=== STOCKOUT GOAL: Fill Rate 96.4% → 99% ===
Current avg fill rate (2022): 96.35%
Target fill rate: 99.00%
Additional revenue if fill_rate → 99%: 17.78M

## 6. Correct return rate calculation and quantify AI size tool impact

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

dataset_path = '/content/drive/MyDrive/dataset/'

# Return rate thực tế từ data chỉ 5-7% — con số 30-36% trước đó bị sai do dùng reviews làm proxy cho orders
# Reviews chỉ có ~20% orders → denominator bị undercount → return rate bị inflate
# Recompute đúng: dùng payments làm denominator (646K orders thực)

returns = pd.read_csv(dataset_path + 'returns.csv')
products = pd.read_csv(dataset_path + 'products.csv')
order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
payments = pd.read_csv(dataset_path + 'payments.csv')
inventory = pd.read_csv(dataset_path + 'inventory.csv')

# TRUE return rate: returns / total unique orders (from payments = most complete)
total_orders = payments['order_id'].nunique()
print(f"Total unique orders (payments): {total_orders:,}")

ret_prod = returns.merge(products[['product_id','segment','category','price','cogs']], on='product_id', how='left')
ret_prod['return_date'] = pd.to_datetime(ret_prod['return_date'])
ret_prod['year'] = ret_prod['return_date'].dt.year

# Overall return rate
print(f"Total returns: {len(returns):,}")
print(f"Overall return rate: {len(returns)/total_orders:.2%}")

# By segment — denominator = order_items unique orders with that segment
oi_seg = order_items.merge(products[['product_id','segment']], on='product_id')\
    .groupby('segment')['order_id'].nunique().reset_index(name='total_orders')
ret_seg = ret_prod.groupby('segment').agg(
    returns=('return_id','count'),
    total_refund=('refund_amount','sum'),
    avg_refund=('refund_amount','mean')
).reset_index()
rr = ret_seg.merge(oi_seg, on='segment')
rr['return_rate'] = rr['returns'] / rr['total_orders']
print("\n=== TRUE RETURN RATE BY SEGMENT ===")
print(rr.sort_values('return_rate', ascending=False).to_string(index=False))

# Revenue impact: if AI size recommendation reduces wrong_size returns by 60%
# wrong_size is #1 return reason for ALL segments
wrong_size_pct = 0.36  # approximate from earlier (733/2126 for Trendy, 4591/13085 for Activewear etc)
# recompute
ret_reason = ret_prod.groupby('return_reason')['return_id'].count().reset_index()
print("\n=== RETURN REASONS ===")
print(ret_reason.sort_values('return_id', ascending=False))
wrong_size_count = ret_reason[ret_reason['return_reason']=='wrong_size']['return_id'].values[0]
wrong_size_pct_actual = wrong_size_count / len(returns)
print(f"\nWrong size: {wrong_size_count:,} = {wrong_size_pct_actual:.1%} of all returns")

# If AI size tool reduces wrong_size returns by 60%
reduction = 0.60
returns_avoided = wrong_size_count * reduction
avg_refund_overall = returns['refund_amount'].mean()
revenue_recovered = returns_avoided * avg_refund_overall
print(f"\n=== GOAL: AI Size Tool reduces wrong_size returns by 60% ===")
print(f"Returns avoided: {returns_avoided:,.0f}")
print(f"Avg refund: {avg_refund_overall:,.0f}")
print(f"Revenue recovered: {revenue_recovered/1e6:.2f}M")

# Streetwear trend
print("\n\n=== STREETWEAR/OUTDOOR TRAJECTORY ===")
# Use inventory units_sold as proxy for demand trend
inv_cat = inventory.groupby(['year','category'])['units_sold'].sum().reset_index()
for cat in ['Streetwear','Outdoor']:
    df = inv_cat[inv_cat['category']==cat].sort_values('year')
    slope, intercept, r, _, _ = stats.linregress(df['year'], df['units_sold'])
    print(f"\n{cat}: {slope:+.0f} units/năm (R²={r**2:.2f})")
    val_2022 = df[df['year']==2022]['units_sold'].values[0]
    print(f"  2022: {val_2022:,} units")
    for yr in [2023, 2025]:
        pred = intercept + slope * yr
        pct_change = (pred - val_2022) / val_2022
        print(f"  {yr}: ~{pred:,.0f} units ({pct_change:+.1%} vs 2022)")
    print(f"  → Nếu không can thiệp, {cat} sẽ về 0 vào năm ~{int(-intercept/slope)}")

Total unique orders (payments): 646,945
Total returns: 39,939
Overall return rate: 6.17%

=== TRUE RETURN RATE BY SEGMENT ===
    segment  returns  total_refund   avg_refund  total_orders  return_rate
 Activewear    13085   65183202.88  4981.521045        171011     0.076516
     Trendy     2126   11145648.17  5242.543824         37054     0.057376
Performance     5442   76399356.88 14038.838089         96303     0.056509
   Everyday    10143  169752795.02 16735.955341        181514     0.055880
    Premium     1722   14762841.67  8573.078786         30944     0.055649
   Standard      601    7208320.83 11993.878253         10924     0.055016
   Balanced     5613  153347776.40 27320.109816        103023     0.054483
All-weather     1207   12798564.70 10603.616156         22553     0.053518

=== RETURN REASONS ===
      return_reason  return_id
4        wrong_size      13967
1         defective       8020
3  not_as_described       7035
0      changed_mind       6931
2     late_delivery 

In [ ]:
import pandas as pd
import numpy as np

dataset_path = '/content/drive/MyDrive/dataset/'

returns = pd.read_csv(dataset_path + 'returns.csv')
order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
customers = pd.read_csv(dataset_path + 'customers.csv')
products = pd.read_csv(dataset_path + 'products.csv')
geography = pd.read_csv(dataset_path + 'geography.csv')
reviews = pd.read_csv(dataset_path + 'reviews.csv')

reviews['review_date'] = pd.to_datetime(reviews['review_date'])
cust_region = customers[['customer_id','zip']].merge(geography[['zip','region']].drop_duplicates(), on='zip', how='left')
order_region = reviews[['order_id','customer_id']].drop_duplicates('order_id')\
    .merge(cust_region, on='customer_id', how='left')

print("Returns sample:")
print(returns.head(3))
print("\nOrder_region sample:")
print(order_region.head(3))
print("\nReturns order_id dtype:", returns['order_id'].dtype)
print("Order_region order_id dtype:", order_region['order_id'].dtype)

# Fix dtype
returns['order_id'] = returns['order_id'].astype(int)
order_region['order_id'] = order_region['order_id'].astype(int)

ret_with_region = returns.merge(order_region[['order_id','region']], on='order_id', how='left')
print("\nReturns with region (non-null region):", ret_with_region['region'].notna().sum(), "of", len(ret_with_region))

# Return rate by region
oi = order_items.merge(products[['product_id','segment']], on='product_id', how='left')
order_seg = oi.groupby('order_id')['segment'].first().reset_index()

ret2 = ret_with_region.merge(order_seg, on='order_id', how='left')

# Orders per region (using reviews as proxy)
orders_per_region = order_region.groupby('region')['order_id'].count().reset_index(name='total_orders')
returns_per_region = ret_with_region.groupby('region')['return_id'].count().reset_index(name='total_returns')
rr = orders_per_region.merge(returns_per_region, on='region', how='left')
rr['return_rate'] = rr['total_returns'] / rr['total_orders']
print("\n=== RETURN RATE BY REGION ===")
print(rr.sort_values('return_rate', ascending=False))

# By segment + region
returns_seg_reg = ret2.groupby(['region','segment'])['return_id'].count().reset_index(name='returns')
orders_seg_reg = order_region.merge(order_seg, on='order_id').groupby(['region','segment'])['order_id'].count().reset_index(name='orders')
rr_seg = returns_seg_reg.merge(orders_seg_reg, on=['region','segment'])
rr_seg['return_rate'] = rr_seg['returns'] / rr_seg['orders']
print("\n=== RETURN RATE BY REGION + SEGMENT ===")
print(rr_seg.sort_values(['region','return_rate'], ascending=[True,False]).to_string())

Returns sample:
    return_id  order_id  product_id return_date  return_reason  \
0  RET-000001         2         609  2012-07-25  late_delivery   
1  RET-000002        32        1862  2012-07-16     wrong_size   
2  RET-000003        35        2359  2012-07-16     wrong_size   

   return_quantity  refund_amount  
0                6       52458.01  
1                2        5141.37  
2                1        5315.95  

Order_region sample:
   order_id  customer_id   zip region
0         1        58578  1109   East
1         3        58811  1473   East
2        10        49101  5262   East

Returns order_id dtype: int64
Order_region order_id dtype: int64

Returns with region (non-null region): 0 of 39939

=== RETURN RATE BY REGION ===
    region  total_orders  total_returns  return_rate
0  Central         31362            NaN          NaN
1     East         50171            NaN          NaN
2     West         29836            NaN          NaN

=== RETURN RATE BY REGION + SEGMENT ===


## 7. Build RFM from DAX logic and quantify segment strategies

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

dataset_path = '/content/drive/MyDrive/dataset/'

order_items = pd.read_csv(dataset_path + 'order_items.csv', low_memory=False)
products = pd.read_csv(dataset_path + 'products.csv')
payments = pd.read_csv(dataset_path + 'payments.csv')
reviews = pd.read_csv(dataset_path + 'reviews.csv')
promotions = pd.read_csv(dataset_path + 'promotions.csv')
customers = pd.read_csv(dataset_path + 'customers.csv')

reviews['review_date'] = pd.to_datetime(reviews['review_date'])
reviews['year'] = reviews['review_date'].dt.year

# ============================================================
# RFM THRESHOLDS từ DAX:
# R: recency <= 1258 → score 30; <= 2283 → 20; else 10
# F: freq >= 3 → 30; >= 2 → 20; else 10
# M: monetary >= 58706 → 30; >= 21068 → 20; else 10
# RFM Score = R + F + M → 30=High, 20-29=Medium, 10-19=Low (average of scores)
# ============================================================

# Build customer RFM from payments + reviews
# Use reviews for recency/frequency, payments for monetary
cust_orders = reviews[['customer_id','order_id','review_date']].drop_duplicates('order_id')
cust_orders = cust_orders.merge(payments[['order_id','payment_value']], on='order_id', how='left')

ref_date = pd.Timestamp('2022-12-31')
rfm = cust_orders.groupby('customer_id').agg(
    recency=('review_date', lambda x: (ref_date - x.max()).days),
    frequency=('order_id','count'),
    monetary=('payment_value','sum')
).reset_index()

# Apply DAX scoring
def r_score(r):
    if r <= 1258: return 30
    elif r <= 2283: return 20
    else: return 10

def f_score(f):
    if f >= 3: return 30
    elif f >= 2: return 20
    else: return 10

def m_score(m):
    if m >= 58706: return 30
    elif m >= 21068: return 20
    else: return 10

rfm['R'] = rfm['recency'].apply(r_score)
rfm['F'] = rfm['frequency'].apply(f_score)
rfm['M'] = rfm['monetary'].apply(m_score)
rfm['RFM_Score'] = (rfm['R'] + rfm['F'] + rfm['M']) / 3

def segment(s):
    if s == 30: return 'High'
    elif s >= 20: return 'Medium'
    else: return 'Low'

rfm['Segment'] = rfm['RFM_Score'].apply(segment)

print("=== RFM SEGMENT DISTRIBUTION ===")
seg_stats = rfm.groupby('Segment').agg(
    customers=('customer_id','count'),
    avg_recency=('recency','mean'),
    avg_frequency=('frequency','mean'),
    avg_monetary=('monetary','mean'),
    total_monetary=('monetary','sum')
).reset_index()
seg_stats['pct_customers'] = seg_stats['customers'] / seg_stats['customers'].sum()
seg_stats['pct_revenue'] = seg_stats['total_monetary'] / seg_stats['total_monetary'].sum()
print(seg_stats.to_string(index=False))

print("\n=== RFM SCORE BREAKDOWN ===")
score_dist = rfm.groupby('RFM_Score').agg(
    customers=('customer_id','count'),
    avg_monetary=('monetary','mean'),
    total_monetary=('monetary','sum')
).reset_index()
score_dist['pct_customers'] = score_dist['customers'] / score_dist['customers'].sum()
score_dist['pct_revenue'] = score_dist['total_monetary'] / score_dist['total_monetary'].sum()
print(score_dist.to_string(index=False))

# ============================================================
# PREDICTIVE: What happens if no action?
# ============================================================
print("\n\n=== PREDICTIVE: Repeat purchase trend ===")
# frequency distribution
print(rfm['frequency'].describe())
print(f"\nCustomers with freq=1: {(rfm['frequency']==1).sum():,} = {(rfm['frequency']==1).mean():.1%}")
print(f"Customers with freq>=2: {(rfm['frequency']>=2).sum():,} = {(rfm['frequency']>=2).mean():.1%}")
print(f"Customers with freq>=3: {(rfm['frequency']>=3).sum():,} = {(rfm['frequency']>=3).mean():.1%}")

# Revenue by frequency group
rfm['freq_group'] = pd.cut(rfm['frequency'], bins=[0,1,2,3,100], labels=['1x','2x','3x','4x+'])
print("\nRevenue by frequency group:")
print(rfm.groupby('freq_group').agg(
    customers=('customer_id','count'),
    total_revenue=('monetary','sum'),
    avg_revenue=('monetary','mean')
).assign(pct_rev=lambda x: x['total_revenue']/x['total_revenue'].sum()).to_string())


# Quantify strategy per RFM segment with promotion analysis
print("\n\n" + "=" * 60)
print("SEGMENT PROFILES & UPGRADE PATH QUANTIFICATION")
print("=" * 60)

# --- HIGH (RFM=30): Retain & Upsell ---
high = rfm[rfm['Segment']=='High']
print(f"\n[HIGH] {len(high):,} customers ({len(high)/len(rfm):.1%})")
print(f"  Avg monetary: {high['monetary'].mean():,.0f}")
print(f"  Avg frequency: {high['frequency'].mean():.1f} lần")
print(f"  Avg recency: {high['recency'].mean():.0f} ngày")
print(f"  Total revenue contribution: {high['monetary'].sum()/1e6:.1f}M")
# If churn 10% of High → loss
churn_loss = high['monetary'].mean() * len(high) * 0.10
print(f"  → If 10% High churn: mất {churn_loss/1e6:.1f}M/năm")
# Goal: increase avg frequency from 5.35 to 6
freq_uplift = (6 - high['frequency'].mean()) * high['monetary'].mean() / high['frequency'].mean()
print(f"  → If avg frequency 5.35→6.0: thêm {freq_uplift * len(high)/1e6:.1f}M")

# --- MEDIUM (RFM 20-27): Upgrade to High ---
med = rfm[rfm['Segment']=='Medium']
print(f"\n[MEDIUM] {len(med):,} customers ({len(med)/len(rfm):.1%})")
print(f"  Avg monetary: {med['monetary'].mean():,.0f}")
print(f"  Avg frequency: {med['frequency'].mean():.1f} lần")
print(f"  Gap to High monetary: {high['monetary'].mean() - med['monetary'].mean():,.0f}")
# Medium → High: if 20% of Medium upgrade
upgrade_pct = 0.20
upgraded = int(len(med) * upgrade_pct)
revenue_gain = upgraded * (high['monetary'].mean() - med['monetary'].mean())
print(f"  → If 20% Medium → High: thêm {revenue_gain/1e6:.1f}M ({upgraded:,} KH)")
# What promotion needed: gap in monetary
monetary_gap = high['monetary'].mean() - med['monetary'].mean()
print(f"  → Monetary gap cần bridge: {monetary_gap:,.0f}")

# Check promotions for relevant ones
print("\n=== PROMOTIONS AVAILABLE ===")
print(promotions[['promo_name','promo_type','discount_value','min_order_value','applicable_category']].to_string(index=False))

# --- LOW (RFM 10-17): Activate or accept churn ---
low = rfm[rfm['Segment']=='Low']
print(f"\n[LOW] {len(low):,} customers ({len(low)/len(rfm):.1%})")
print(f"  Avg monetary: {low['monetary'].mean():,.0f}")
print(f"  Avg frequency: {low['frequency'].mean():.1f} lần")
print(f"  Total revenue: {low['monetary'].sum()/1e6:.1f}M ({low['monetary'].sum()/rfm['monetary'].sum():.1%})")
# Goal: convert 15% of Low to Medium
convert_pct = 0.15
converted = int(len(low) * convert_pct)
rev_gain_low = converted * (med['monetary'].mean() - low['monetary'].mean())
print(f"  → If 15% Low → Medium: thêm {rev_gain_low/1e6:.1f}M ({converted:,} KH)")

# ============================================================
# PREDICTIVE: What if no action — revenue trajectory
# ============================================================
print("\n\n=== PREDICTIVE: Revenue at risk if no loyalty action ===")
# Annual new customers acquiring vs losing High
# From data: avg recency of High = 559 days → still active
# But 48.1% of all customers only bought once → these are likely churning
one_time = rfm[rfm['frequency']==1]
print(f"One-time buyers: {len(one_time):,} = {len(one_time)/len(rfm):.1%}")
print(f"Revenue from one-time buyers: {one_time['monetary'].sum()/1e6:.1f}M ({one_time['monetary'].sum()/rfm['monetary'].sum():.1%})")
print(f"If these never return → lost potential LTV vs repeat buyers:")
repeat_avg = rfm[rfm['frequency']>=2]['monetary'].mean()
one_time_avg = one_time['monetary'].mean()
print(f"  Avg LTV repeat buyer: {repeat_avg:,.0f}")
print(f"  Avg LTV one-time: {one_time_avg:,.0f}")
print(f"  LTV gap per customer: {repeat_avg - one_time_avg:,.0f}")
print(f"  Total untapped LTV if 20% one-time → repeat: {len(one_time)*0.20*(repeat_avg-one_time_avg)/1e6:.1f}M")